# tiny_model

> Phase 12 test support: `tinyModel(dir)` builds a tiny random Qwen2-style model and a byte-level BPE tokenizer with a chat template (no download), so the training and generation code can be smoke-tested on any machine with the `[train]` extra.

In [ ]:
#| default_exp tiny_model

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
def tinyModel(d, vocab=400):
    "A tiny random Qwen2-style model with a byte-level BPE tokenizer and a chat template, saved to d (no download; smoke tests)."
    from pathlib import Path
    from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers
    from transformers import PreTrainedTokenizerFast, Qwen2Config, Qwen2ForCausalLM
    import torch
    d = Path(d)
    if (d / 'config.json').exists():
        return str(d)
    from explRL.clue_text import TextClueEnv
    from explRL.clueboard import ClueBoardEnv
    corpus = [TextClueEnv(ClueBoardEnv(0, 4, m=2, r=1, R=30)).rules(), '{"notes": "x mod 7 = 2", "action": "open", "target": "T1", "answer": 0, "clue": ""}'] * 3
    tk = Tokenizer(models.BPE(unk_token='<unk>'))
    tk.pre_tokenizer, tk.decoder = pre_tokenizers.ByteLevel(add_prefix_space=False), decoders.ByteLevel()
    tk.train_from_iterator(corpus, trainers.BpeTrainer(vocab_size=vocab, special_tokens=['<unk>', '<pad>', '<eos>', '<s>'],
                                                       initial_alphabet=pre_tokenizers.ByteLevel.alphabet()))
    tok = PreTrainedTokenizerFast(tokenizer_object=tk, unk_token='<unk>', pad_token='<pad>', eos_token='<eos>', bos_token='<s>')
    tok.chat_template = ("{% for m in messages %}<s>{{ m['role'] }}\n{{ m['content'] }}<eos>{% endfor %}"
                         "{% if add_generation_prompt %}<s>assistant\n{% endif %}")
    torch.manual_seed(0)
    cfg = Qwen2Config(vocab_size=len(tok), hidden_size=64, intermediate_size=128, num_hidden_layers=2, num_attention_heads=4,
                      num_key_value_heads=2, max_position_embeddings=4096, eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id,
                      bos_token_id=tok.bos_token_id, tie_word_embeddings=True)
    Qwen2ForCausalLM(cfg).save_pretrained(d)
    tok.save_pretrained(d)
    return str(d)

## Tests

In [ ]:
import inspect
from explRL.tiny_model import tinyModel
assert 'Qwen2Config' in inspect.getsource(tinyModel)        # the model itself needs the [train] extra (see the train cell)

In [ ]:
#| train
import tempfile
from explRL.local_policy import LocalPolicy
with tempfile.TemporaryDirectory() as d:
    m = tinyModel(d)
    assert tinyModel(d) == m                                  # built once
    from explRL.clue_policy import renderPrompt
    from explRL.clue_text import TextClueEnv
    from explRL.clueboard import ClueBoardEnv
    env = TextClueEnv(ClueBoardEnv(0, 2, m=2, r=1, R=30))
    assert len(LocalPolicy(m, maxNew=8, device='cpu')([renderPrompt(env, env.observe(0, 0.0))])) == 1

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()